### History matching example
#### py scripts are run as casper jobs to scale sampling

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import os, json, sys
import tensorflow as tf
from scipy.stats import qmc

2026-01-22 15:03:39.590775: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-22 15:03:41.062379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769119421.415382   67047 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769119421.519466   67047 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-22 15:03:42.269147: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [1]:
utils_dir = '../utils/'
from utils.hm_pyfunctions import *

2026-11-23 16:51:03.926738: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-23 16:51:03.943416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1795477863.959499   46818 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795477863.965324   46818 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-23 16:51:03.984494: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [111]:
# user mods
n_universal_samples = 500
n_pft_samples = 1000
n_pfts = 14

I_tolerance = 3.0 # implausibility tolerance

out_dir = '/glade/work/linnia/CLM6-PPE/ctsm53065_wave3/NROY/'


### Prepare inputs

In [112]:
# get config information
utils_dir = '../utils/'

with open(os.path.join(utils_dir, "param_info.json"), "r") as f:
    param_info = json.load(f)

with open(os.path.join(utils_dir, "biome_info.json"), "r") as f:
    biome_info = json.load(f)

universal_param_names = param_info['u_params']
pft_param_names = param_info['pft_params']
n_universal_params = len(param_info['u_params'])
n_pft_params = len(param_info['pft_params']) # Number of parameters per PFT


In [113]:
# load observational data
obs = xr.open_dataset(utils_dir + 'obs_statistics_byBiome.nc')

In [115]:
# stack up all your observational data and emulators for each input metric (variable and biome)
paths = {
    'LAI': './emulators/biome_lai_',
    'GPP': './emulators/biome_gpp_',
    'TVC': './emulators/biome_tvc_',
    'LH' : './emulators/biome_lh_'
}

biomes = [1,2,3,4,5,6,7,8,9,10,11,12,13]
obs_mean = []
obs_var = []
emulators = [] 
biome_list = [] 

var_list = ['LAI','GPP','TVC','LH']
for v in var_list:
    for b in biomes:
        biome_list.append(str(b))
    
        obs_mean.append(obs[v+'_mean'].isel(biome=b).values)
        obs_var.append(obs[v+'_stdev'].isel(biome=b).values**2)
    
        emulators.append(tf.saved_model.load(paths[v]+str(b)))

### History Matching

In [116]:
# create universal sample
sampler = qmc.LatinHypercube(d=n_universal_params)
s = sampler.random(n=n_universal_samples)
universal_parameter_samples = pd.DataFrame(s,columns=universal_param_names)

In [117]:
l_bounds = np.zeros(n_universal_params)
u_bounds = np.ones(n_universal_params)
u_bounds[universal_param_names.index('RF_SS')] = 0.2
scaled_s = qmc.scale(s, l_bounds, u_bounds)
universal_parameter_samples = pd.DataFrame(scaled_s,columns=universal_param_names)

In [122]:
%%time
idx = 3
all_nroy_param_sets_from_universal = [] # List to collect NROY sets from each universal sample

# Loop over each universal parameter sample
for u_idx in range(len(universal_parameter_samples)):
    # create sample (dataframe) for the current universal parameter set
    usample_values = np.tile(universal_parameter_samples.iloc[[u_idx]].values,(n_pft_samples,1))
    psample_values = np.random.rand(n_pft_samples,n_pfts*n_pft_params)
    combined_values = np.concatenate([usample_values,psample_values],axis=1)

    all_pft_param_column_names = []
    for pft_id in range(1, n_pfts + 1):
        for param_name in pft_param_names:
            all_pft_param_column_names.append(f"pft{pft_id}_{param_name}")
    all_columns = universal_param_names + all_pft_param_column_names
    sample_df = pd.DataFrame(combined_values, columns=all_columns)

    # History matching for the current universal parameter set
    num_metrics = len(emulators)
    all_implausibilities_for_samples = np.empty((n_pft_samples, num_metrics))

    for metric_ix in range(num_metrics):
        biome_id = str(biome_list[metric_ix]) # Ensure biome_id is an integer for dict lookup
        emulator_object = emulators[metric_ix] # access emulator from a list/array

        y_pred, y_pred_var = call_biome_emulator(sample_df, biome_id, emulator_object, biome_info)

        # calc_Implausibility 
        metric_I = calc_Implausibility(y_pred, y_pred_var, obs_mean[metric_ix], obs_var[metric_ix])
        
        # Store the implausibility values for the current biome/metric
        all_implausibilities_for_samples[:, metric_ix] = metric_I.squeeze()

    # identify parameter sets that are Not Ruled Out Yet for the current universal sample
    is_nroy_mask = (all_implausibilities_for_samples < I_tolerance).all(axis=1)
    current_u_nroy_param_sets = sample_df[is_nroy_mask]
    
    # Append the NROY sets found for this universal parameter to the master list
    all_nroy_param_sets_from_universal.append(current_u_nroy_param_sets)

# Concatenate all NROY_param_sets from all universal samples
NROY_param_sets = pd.concat(all_nroy_param_sets_from_universal).reset_index(drop=True)
out_file = out_dir+'NROY_param_sets_'+str(idx)+'.csv'
NROY_param_sets.to_csv(out_file,index=False)
all_nroy_param_sets_from_universal = []


CPU times: user 22min 14s, sys: 8.81 s, total: 22min 22s
Wall time: 24min 50s
